# Tutorial 01 — The Data Model & the Math of LCA

Companion explainer: **01_ecosystem_and_lca_math.md**. We compute a tiny
3-process system **by hand with NumPy**, then build the *same* system in
Brightway and show the score matches to machine precision. This trust-but-
verify pattern is the backbone of every model you'll build.

In [1]:
import numpy as np
import bw2data as bd
import bw2calc as bc

bd.projects.set_current("bw25-tutorials")
BIOSPHERE = next(d for d in bd.databases if "biosphere" in d.lower())
bio = bd.Database(BIOSPHERE)

## The system

| Process (1 unit =) | Inputs | Emissions |
|---|---|---|
| electricity (1 kWh) | – | 0.9 kg CO2 |
| steel (1 kg) | 2.5 kWh electricity | 1.8 kg CO2 |
| widget (1 kg) | 0.5 kg steel, 1.0 kWh electricity | – |

## Step 1 — solve it by hand

Columns/rows ordered [elec, steel, widget]. A is the technosphere matrix
(production positive on the diagonal, inputs negative), B the biosphere row.

In [2]:
A = np.array([
    [1.0, -2.5, -1.0],   # electricity balance
    [0.0,  1.0, -0.5],   # steel balance
    [0.0,  0.0,  1.0],   # widget balance
])
B = np.array([[0.9, 1.8, 0.0]])  # kg CO2 per unit of each process

f = np.array([0.0, 0.0, 1.0])    # functional unit: 1 widget
s = np.linalg.solve(A, f)        # scaling vector
g = B @ s                        # total inventory (kg CO2)
print("scaling vector s =", s)
print("inventory g (kg CO2) =", g)
hand_score = float(g[0])         # CF for CO2 under GWP100 = 1
print("hand-calculated GWP =", hand_score, "kg CO2-eq")

scaling vector s =

[2.25 0.5  1.  ]

inventory g (kg CO2) =

[2.925]

hand-calculated GWP =

2.925

kg CO2-eq

## Step 2 — build the same system in Brightway

In [3]:
co2 = next(f for f in bio
           if f["name"] == "Carbon dioxide, fossil" and f["categories"] == ("air",))

DB = "t01_widget"
if DB in bd.databases:
    del bd.databases[DB]

bd.Database(DB).write({
    (DB, "electricity"): {
        "name": "electricity production", "unit": "kilowatt hour",
        "exchanges": [
            {"input": (DB, "electricity"), "amount": 1.0, "type": "production"},
            {"input": co2.key, "amount": 0.9, "type": "biosphere"},
        ],
    },
    (DB, "steel"): {
        "name": "steel production", "unit": "kilogram",
        "exchanges": [
            {"input": (DB, "steel"), "amount": 1.0, "type": "production"},
            {"input": (DB, "electricity"), "amount": 2.5, "type": "technosphere"},
            {"input": co2.key, "amount": 1.8, "type": "biosphere"},
        ],
    },
    (DB, "widget"): {
        "name": "widget production", "unit": "kilogram",
        "exchanges": [
            {"input": (DB, "widget"), "amount": 1.0, "type": "production"},
            {"input": (DB, "steel"), "amount": 0.5, "type": "technosphere"},
            {"input": (DB, "electricity"), "amount": 1.0, "type": "technosphere"},
        ],
    },
})
widget = bd.get_node(database=DB, code="widget")
print("built:", widget)

13:15:20-0400

 [

warning  

] 

Not able to determine geocollections for all datasets. This database is not ready for regionalization.

...

info     

] 

Vacuuming database            

built:

'widget production' (kilogram, None, None)

## Step 3 — calculate with a real GWP method and compare

In [4]:
gwp = next(m for m in bd.methods
           if "IPCC 2013" in str(m) and "GWP100" in str(m).replace(" ", "")
           and "no LT" not in str(m) and "SLCF" not in str(m))

lca = bc.LCA({widget: 1}, method=gwp)
lca.lci()
lca.lcia()
print("Brightway score =", lca.score, "kg CO2-eq")
print("hand score      =", hand_score, "kg CO2-eq")
print("relative diff   =", abs(lca.score - hand_score) / hand_score)
# Brightway stores exchange amounts as float32 in the datapackage, so agreement
# is to ~float32 precision (~1e-7 relative), not float64. That tiny gap IS the
# lesson: the framework is exact given its stored data; don't chase 1e-15.
assert abs(lca.score - hand_score) / hand_score < 1e-5, "hand vs Brightway mismatch!"
print("✅ match to float32 precision (Brightway stores amounts as float32)")

Brightway score =

2.924999922513962

kg CO2-eq

hand score      =

2.925

kg CO2-eq

relative diff   =

2.6490953172776798e-08

✅ match to float32 precision (Brightway stores amounts as float32)

## Step 4 — peek at Brightway's internal matrices
The A and B we typed by hand *are* what Brightway assembled. Map matrix indices
back to activities via `lca.dicts`.

In [5]:
A_bw = np.asarray(lca.technosphere_matrix.todense())
print("technosphere matrix (rows=cols=processes):")
order = {v: bd.get_activity(k)["name"] for k, v in lca.activity_dict.items()} \
    if hasattr(lca, "activity_dict") else None
print(np.round(A_bw, 3))
print("\nsupply array s =", np.round(lca.supply_array, 4))
print("characterized inventory total =", lca.characterized_inventory.sum())

technosphere matrix (rows=cols=processes):

[[ 1.  -2.5 -1. ]
 [ 0.   1.  -0.5]
 [ 0.   0.   1. ]]


supply array s =

[2.25 0.5  1.  ]

characterized inventory total =

2.924999922513962

Note Brightway may order rows/columns differently than our hand matrix — the
*score* is invariant to ordering, which is exactly why we compare scores, not
raw matrices. Next: **02 — projects & data management**.